In [0]:
%sql
CREATE OR REPLACE TABLE main.gh_archive.silver_events AS
SELECT
    id AS event_id,
    type,
    created_at,
    actor.id AS actor_id,
    actor.display_login AS display_login,
    repo.id AS repo_id,
    repo.name AS repo_name,
    repo.url AS repo_url,
    payload.action AS action,
    payload.ref AS raw_ref,
    ELEMENT_AT(SPLIT(payload.ref, '/'), -1) AS branch_name,
    payload.pull_request.number AS pr_number
FROM main.gh_archive.bronze_events
WHERE type IN ('PushEvent', 'PullRequestEvent', 'IssueCommentEvent')

In [0]:
%sql
CREATE OR REPLACE TABLE main.gh_archive.silver_events_quarantine AS
SELECT
    event_id,
    type,
    actor_id,
    repo_id, 
    pr_number
FROM main.gh_archive.silver_events
WHERE (event_id IS NULL OR type IS NULL OR actor_id IS NULL OR repo_id IS NULL)
OR  (type = 'PullRequestEvent' AND pr_number IS NULL);

In [0]:
%python
quarantine_count = spark.table("main.gh_archive.silver_events_quarantine").count()
print(f"Quarantined rows this run: {quarantine_count}")
if quarantine_count > 0:
    print("WARNING: Quarantined rows detected — review silver_events_quarantine for data quality issues.")